# 07 · Baseline Modeling

**Purpose:** Train a first RandomForest model on news features to predict
next-day stock returns.

Two tasks are trained:
- **Classification** — predict `direction` (up / down)
- **Regression** — predict `next_day_return`

The train/test split is **time-based** (no shuffling) to avoid look-ahead leakage.

**Input:** `modeling_dataset.parquet`  
**Output:** `data/models/rf_classifier.pkl`, `data/models/rf_regressor.pkl`, `test_predictions.parquet`

In [1]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, mean_squared_error,
)

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save, DATA_DIR

MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

In [2]:
dataset = load("modeling_dataset")
print(f"Rows: {len(dataset):,}  |  Tickers: {dataset['ticker'].nunique()}")

Rows: 175  |  Tickers: 32


## 1 · Feature selection

In [3]:
BASE_FEATURES = [
    "news_count", "avg_sentiment", "std_sentiment",
    "positive_ratio", "negative_ratio",
    "event_ma", "event_earnings", "event_management", "event_legal",
    "negated_count", "speculative_count",
]

ROLLING_FEATURES = [
    f"{col}_r{w}d"
    for col in ["news_count", "avg_sentiment", "std_sentiment",
                "positive_ratio", "negative_ratio",
                "event_ma", "event_earnings", "event_management", "event_legal"]
    for w in [3, 7]
]

FEATURE_COLS = [c for c in BASE_FEATURES + ROLLING_FEATURES if c in dataset.columns]

TARGET_CLS = "direction"
TARGET_REG = "next_day_return"

print(f"Using {len(FEATURE_COLS)} features")

# Only keep rows where targets are available and features non-null
df = dataset.dropna(subset=[TARGET_CLS, TARGET_REG] + FEATURE_COLS).copy()
df[TARGET_CLS] = df[TARGET_CLS].astype(int)
print(f"Usable rows: {len(df):,}")

Using 29 features
Usable rows: 175


## 2 · Time-based train / test split

In [4]:
TEST_RATIO = 0.20

df = df.sort_values("date")
cutoff = df["date"].quantile(1 - TEST_RATIO)

train = df[df["date"] <  cutoff]
test  = df[df["date"] >= cutoff]

print(f"Train: {len(train):,} rows  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test:  {len(test):,}  rows  ({test['date'].min().date()}  → {test['date'].max().date()})")
print(f"\nClass balance — train: {train[TARGET_CLS].mean():.1%} up  |  test: {test[TARGET_CLS].mean():.1%} up")

X_train, X_test = train[FEATURE_COLS], test[FEATURE_COLS]
y_cls_train, y_cls_test = train[TARGET_CLS], test[TARGET_CLS]
y_reg_train, y_reg_test = train[TARGET_REG], test[TARGET_REG]

Train: 130 rows  (2022-04-22 → 2026-04-22)
Test:  45  rows  (2026-04-23  → 2026-04-24)

Class balance — train: 41.5% up  |  test: 33.3% up


## 3 · Classification (direction)

In [5]:
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_cls_train)

y_pred_cls = clf.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_cls_test, y_pred_cls):.3f}")
print(f"(Baseline: {max(y_cls_test.mean(), 1 - y_cls_test.mean()):.3f} — always-majority)\n")
print(classification_report(y_cls_test, y_pred_cls, target_names=["Down", "Up"]))

Accuracy:  0.444
(Baseline: 0.667 — always-majority)

              precision    recall  f1-score   support

        Down       0.86      0.20      0.32        30
          Up       0.37      0.93      0.53        15

    accuracy                           0.44        45
   macro avg       0.61      0.57      0.43        45
weighted avg       0.69      0.44      0.39        45



## 4 · Regression (next-day return)

In [6]:
reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)
reg.fit(X_train, y_reg_train)

y_pred_reg = reg.predict(X_test)

rmse     = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
null_rmse = np.sqrt(mean_squared_error(y_reg_test, np.zeros_like(y_reg_test)))  # predict-zero baseline
dir_acc  = np.mean(np.sign(y_pred_reg) == np.sign(y_reg_test))

print(f"RMSE (model):     {rmse:.5f}")
print(f"RMSE (zero pred): {null_rmse:.5f}")
print(f"Directional acc:  {dir_acc:.3f}")

RMSE (model):     0.01931
RMSE (zero pred): 0.01860
Directional acc:  0.422


## 5 · Save models and predictions

In [7]:
with open(MODELS_DIR / "rf_classifier.pkl", "wb") as f:
    pickle.dump((clf, FEATURE_COLS), f)

with open(MODELS_DIR / "rf_regressor.pkl", "wb") as f:
    pickle.dump((reg, FEATURE_COLS), f)

print("Models saved to", MODELS_DIR)

test_preds = test[["ticker", "date", TARGET_CLS, TARGET_REG]].copy()
test_preds["pred_direction"] = y_pred_cls
test_preds["pred_return"]    = y_pred_reg

save(test_preds, "test_predictions")

Models saved to C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\data\models
  saved 45 rows  →  test_predictions.parquet


WindowsPath('C:/_PROJECTS/pfa_bvc/Notebooks/signal_pipeline/data/test_predictions.parquet')